# Book Recommendation System (Content-Based)
### Presentation Notebook for Applied AI (Mathematics Class)

This notebook builds a simple recommender system for books using:
- **TF-IDF vectors** (text -> numbers)
- **Cosine similarity** (angle between vectors)
- A straightforward ranking rule

## Roadmap
1. Load and prepare book data
2. Build a mathematical representation of each book
3. Measure similarity with cosine similarity
4. Recommend top books with clean, readable outputs

### Import Libraries

In [1]:
# Imports
import numpy as np
import pandas as pd
import os
import difflib
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Pandas display settings
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

## 1. Data Preparation

### Load Data and Snapshot

In [2]:
# Load dataset
csv_path = "data.csv"
df = pd.read_csv(csv_path)

# Keep needed columns
cols = ["title", "authors", "categories", "published_year"]
df = df[cols].copy()

df.head(5)

,title,authors,categories,published_year
0,Gilead,Marilynne Robinson,Fiction,2004.0
1,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,2000.0
2,The One Tree,Stephen R. Donaldson,American fiction,1982.0
3,Rage of angels,Sidney Sheldon,Fiction,1993.0
4,The Four Loves,Clive Staples Lewis,Christian life,2002.0


In [3]:
# Dataset summary
summary = pd.DataFrame({ "Metric": ["Rows (books)", "Columns", "Missing values"],"Value": [len(df), df.shape[1], int(df.isna().sum().sum())],})
summary

,Metric,Value
0,Rows (books),6810
1,Columns,4
2,Missing values,177


### Feature Englineering
We turn each book into one combined text string called a book profile

In [4]:
# Features used
selected_features = ["title", "authors", "categories", "published_year"]

feature_table = pd.DataFrame({"Feature": selected_features, "Role": ["Keywords", "Author", "Genre", "Year"]})
feature_table

,Feature,Role
0,title,Keywords
1,authors,Author
2,categories,Genre
3,published_year,Year


### Data Cleaning and Profile Construction

In [5]:
# Fill missing + build one text profile per book
df[selected_features] = df[selected_features].fillna("") 
df["published_year"] = df["published_year"].astype(str).str.replace(".0", "", regex=False)
df["book_profile"] = df[selected_features].agg(" ".join, axis=1)

print("Book profiles created.")
df[["title", "book_profile"]].head(5)

Book profiles created.


,title,book_profile
0,Gilead,Gilead Marilynne Robinson Fiction 2004
1,Spider's Web,Spider's Web Charles Osborne;Agatha Christie Detective and mystery stories 2000
2,The One Tree,The One Tree Stephen R. Donaldson American fiction 1982
3,Rage of angels,Rage of angels Sidney Sheldon Fiction 1993
4,The Four Loves,The Four Loves Clive Staples Lewis Christian life 2002


In [6]:
# Verify no missing values remain in model features
missing_check = df[selected_features + ["book_profile"]].isna().sum().reset_index()
missing_check.columns = ["Column", "Missing values"]
missing_check

,Column,Missing values
0,title,0
1,authors,0
2,categories,0
3,published_year,0
4,book_profile,0


## 2. Build the Recommender Model

In [7]:
# TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")
feature_vectors = vectorizer.fit_transform(df["book_profile"])

n_books, n_terms = feature_vectors.shape
density = feature_vectors.nnz / (n_books * n_terms) #Number of non-zero values
sparsity = 1 - density #Number of zero vaslues

model_stats = pd.DataFrame({"Quantity": ["Books", "Vocabulary", "Density", "Sparsity"],"Value": [n_books, n_terms, f"{density:.4f}", f"{sparsity:.4f}"],})
model_stats

,Quantity,Value
0,Books,6810
1,Vocabulary,10364
2,Density,0.0007
3,Sparsity,0.9993


In [8]:
# Check top TF-IDF terms for one book
example_title = "Rage of Angels"
titles_lower = df["title"].str.lower()

example_idx = titles_lower[titles_lower == example_title.lower()].index[0] if (titles_lower == example_title.lower()).any() else 0
example_title = df.loc[example_idx, "title"]

row = feature_vectors[example_idx].toarray().ravel()
top_idx = row.argsort()[-8:][::-1] 

top_terms = pd.DataFrame({"Term": vectorizer.get_feature_names_out()[top_idx],"TF-IDF weight": np.round(row[top_idx], 4),})

print("Example book used:", example_title)
top_terms

Example book used: Rage of angels


,Term,TF-IDF weight
0,rage,0.5287
1,sidney,0.4693
2,sheldon,0.4448
3,angels,0.4448
4,1993,0.3058
5,fiction,0.1052
6,владимир,0.0000
7,1491,0.0000


### Cosine Similarity

In [9]:
# Cosine similarity between all books
similarity = cosine_similarity(feature_vectors)
print(f"Similarity matrix shape: {similarity.shape}")

Similarity matrix shape: (6810, 6810)


### Recommendation Test and Demo

In [10]:
list_of_all_titles = df["title"].tolist()

def recommend_books_input(top_n=10):
    book_name = input("Enter your favourite book name: ").strip()
    if not book_name:
        print("No input given.")
        return

    match = difflib.get_close_matches(book_name, list_of_all_titles, n=1, cutoff=0.4)
    if not match:
        print("No close match found.")
        return

    close_match = match[0]
    idx = df.index[df["title"] == close_match][0]

    recs = pd.DataFrame({"title": df["title"], "score": similarity[idx]}).drop(index=idx)
    recs = recs.sort_values("score", ascending=False).drop_duplicates("title").head(top_n).reset_index(drop=True)

    print("\nClosest title in dataset:", close_match)
    print("\nBooks suggested for you:\n")
    for i, t in enumerate(recs["title"], start=1):
        print(f"{i}. {t}")

# Run
top_n = int(input("How many similar books do you want to see? ").strip() or "10")
recommend_books_input(top_n=top_n)

How many similar books do you want to see?  10
Enter your favourite book name:  Rage of angels



Closest title in dataset: Rage of angels

Books suggested for you:

1. Tell Me Your Dreams
2. The Sky Is Falling
3. Nothing Lasts Forever
4. Master of the Game
5. If Tomorrow Comes
6. Gold Rage
7. What Angels Fear
8. Angels & Demons
9. My Perfect Life
10. Angels
